# Chapter 01 — MLflow Foundations for Senior SREs

This chapter is the single consolidated foundation file for MLflow basics. It combines the introductory and getting-started material into one coherent study guide without repeating the same ideas in multiple files.

The goal is to understand MLflow as an operational layer around machine learning work: experiment tracking, model artifact management, reproducibility, model comparison, and production governance.

Use this notebook as both reading material and a hands-on practice notebook. Read the theory sections first, then run the code cells locally and inspect the MLflow UI.


# 1. The problem MLflow solves

Machine learning projects do not only produce source code. They produce trained model files, metrics, plots, evaluation reports, feature logic, preprocessing steps, and deployment candidates. Without tracking, teams quickly lose answers to important questions:

- Which run produced this model?
- Which parameters were used?
- Which dataset or feature version was used?
- What metrics did the model achieve?
- Where is the trained model artifact stored?
- Can another engineer reproduce this result?
- Can we roll back to a previous model version?

From an SRE perspective, this is similar to the discipline we already expect from software delivery. You would not deploy an unknown binary without knowing the source commit, build pipeline, dependencies, test result, artifact location, and rollback path. A production ML model deserves the same discipline.

MLflow provides that discipline for ML workflows. It gives a standard way to track experiments, log parameters and metrics, store artifacts, package models, compare runs, and manage model lifecycle metadata.


# 2. ML lifecycle and where MLflow fits

A typical ML lifecycle looks like this:

```text
source data
  -> data preparation
  -> exploratory data analysis
  -> feature engineering
  -> training
  -> validation
  -> model packaging
  -> model registration
  -> deployment
  -> monitoring
  -> retraining
```

MLflow does not replace all tools in that lifecycle. It is not a scheduler, not Kubernetes, not a data warehouse, and not a full monitoring platform. It is best understood as the metadata and artifact control layer around model development.

| Stage | Typical question | MLflow contribution |
|---|---|---|
| Data preparation | Which data was used? | Track dataset references, schema notes, data version tags |
| Feature engineering | Which transformations were used? | Track preprocessing parameters and feature artifacts |
| Training | Which model and parameters were tried? | Log runs, parameters, metrics, tags |
| Validation | Which run performed best? | Compare runs and metrics |
| Packaging | Where is the trained model? | Store model artifacts in a standard format |
| Registry | Which version is approved? | Manage registered models and versions |
| Deployment | Which model is deployed? | Link deployments to model URI/version |
| Monitoring | Which version is behaving badly? | Connect runtime metrics back to model metadata |


# 3. Core vocabulary

## Experiment
An experiment is a container for related runs. Example: all attempts to build an Iris classifier can live inside one experiment.

## Run
A run is one execution of a training, validation, or evaluation job. Every run gets a unique run ID.

## Parameters
Parameters are input configuration values such as model type, learning rate, max depth, solver, random seed, test split, and preprocessing strategy.

## Metrics
Metrics are measured outputs such as accuracy, precision, recall, F1 score, RMSE, loss, latency, and model size. Metrics are used to compare runs and implement quality gates.

## Tags
Tags are searchable metadata such as team, owner, environment, Git commit, branch, CI build ID, dataset version, and risk classification. Tags matter for governance and automation.

## Artifacts
Artifacts are files produced by a run: model files, plots, reports, model cards, dependency files, validation outputs, and sample payloads.

## Model
An MLflow model is a packaged model artifact that can be loaded consistently. MLflow supports multiple model flavors such as scikit-learn, PyTorch, TensorFlow, Keras, and generic Python function models.

## Model registry
The model registry manages named models, versions, aliases, descriptions, and lifecycle metadata. This is where promotion, approval, and rollback workflows become important.


# 4. DevOps-to-MLflow mapping

| Software delivery concept | MLflow concept | Meaning |
|---|---|---|
| Git commit | Run metadata | Record of what was executed |
| CI build | Training run | One execution that produces output |
| Build parameters | MLflow parameters | Input configuration for the run |
| Test results | MLflow metrics | Quality signals |
| Build artifact | MLflow artifact/model | Output of the run |
| Artifact repository | Artifact store | Durable location for produced files |
| Release candidate | Registered model version | Candidate for promotion |
| Staging/production | Registry alias or lifecycle state | Controlled model lifecycle |
| Rollback | Previous model version | Return to known-good behavior |

A useful interview sentence: MLflow does for ML experiments and model artifacts what CI/CD and artifact repositories do for software builds: it records what was run, what inputs were used, what outputs were produced, and which artifact is eligible for promotion.


# 5. Local practice setup

Create or reuse the local environment for this repository:

```bash
cd /Users/jithinpjoseph/Documents/GitHub/SRE-Challenges/mlops/mlflow
python3 -m venv .venv
source .venv/bin/activate
pip install mlflow scikit-learn pandas matplotlib joblib
python -m ipykernel install --user --name sre-challenges-mlops --display-name "SRE Challenges MLOps"
jupyter lab
```

After running the notebook cells, open the MLflow UI:

```bash
mlflow ui --backend-store-uri ./mlruns --host 127.0.0.1 --port 5000
```

Then open `http://127.0.0.1:5000`.

Local `mlruns` storage is fine for learning. It is not a production architecture.


In [ ]:
# Optional dependency installation. Run once if your kernel does not have these libraries.
# !pip install -q mlflow scikit-learn pandas matplotlib joblib


In [ ]:
from pathlib import Path
import json
import platform
import sys
from datetime import datetime, timezone

import mlflow
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

print('Python:', sys.version)
print('Platform:', platform.platform())
print('MLflow version:', mlflow.__version__)


# 6. Configure local MLflow tracking

MLflow needs a tracking URI. For this chapter, we store runs in a local `mlruns` directory. Later, a production setup should use a remote tracking server, database backend, and durable artifact store.


In [ ]:
tracking_dir = Path('mlruns').resolve()
mlflow.set_tracking_uri(tracking_dir.as_uri())

experiment_name = 'chapter-01-mlflow-foundations'
mlflow.set_experiment(experiment_name)

print('Tracking URI:', mlflow.get_tracking_uri())
print('Experiment:', experiment_name)


# 7. First run: log parameters, metrics, tags, and an artifact

This first run does not train a real model. It teaches the basic shape of MLflow tracking. Think of it as a CI job that records metadata and produces a report.


In [ ]:
report = {
    'chapter': 'mlflow-foundations',
    'purpose': 'learn basic MLflow tracking concepts',
    'created_at': datetime.now(timezone.utc).isoformat(),
    'notes': [
        'This is a simple learning run.',
        'It demonstrates parameters, metrics, tags, and artifacts.',
        'The next sections train actual models.'
    ],
}

report_path = Path('basic_tracking_report.json')
report_path.write_text(json.dumps(report, indent=2))

with mlflow.start_run(run_name='basic-tracking-demo') as run:
    mlflow.log_param('workflow_type', 'intro-demo')
    mlflow.log_param('model_type', 'none')
    mlflow.log_param('dataset', 'none')
    mlflow.log_param('random_seed', 42)

    mlflow.log_metric('example_accuracy', 0.91)
    mlflow.log_metric('example_f1_score', 0.89)

    mlflow.set_tag('team', 'sre-mlops-learning')
    mlflow.set_tag('owner_role', 'senior-sre')
    mlflow.set_tag('environment', 'local')
    mlflow.set_tag('chapter', 'mlflow-foundations')

    mlflow.log_artifact(str(report_path), artifact_path='reports')

    print('Run ID:', run.info.run_id)

report_path.unlink(missing_ok=True)


# 8. Train and log real models

Now we train two simple classifiers on the Iris dataset. The goal is not the dataset. The goal is to learn the repeatable MLflow workflow: train, evaluate, log parameters, log metrics, log artifacts, log model, compare runs, and load the selected model for inference.


In [ ]:
iris = load_iris(as_frame=True)
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

def log_confusion_matrix_artifact(y_true, y_pred, labels, filename='confusion_matrix.png'):
    fig, ax = plt.subplots(figsize=(6, 4))
    cm = confusion_matrix(y_true, y_pred)
    display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    display.plot(ax=ax)
    ax.set_title('Confusion Matrix')
    fig.tight_layout()
    output_path = Path(filename)
    fig.savefig(output_path)
    plt.close(fig)
    mlflow.log_artifact(str(output_path), artifact_path='plots')
    output_path.unlink(missing_ok=True)

def evaluate_classifier(model, X_test, y_test):
    predictions = model.predict(X_test)
    return {
        'accuracy': accuracy_score(y_test, predictions),
        'precision_macro': precision_score(y_test, predictions, average='macro'),
        'recall_macro': recall_score(y_test, predictions, average='macro'),
        'f1_macro': f1_score(y_test, predictions, average='macro'),
    }, predictions

print('Training rows:', len(X_train))
print('Test rows:', len(X_test))
print('Features:', list(X.columns))


In [ ]:
with mlflow.start_run(run_name='logistic-regression-baseline') as run:
    params = {
        'model_type': 'LogisticRegression',
        'solver': 'lbfgs',
        'max_iter': 200,
        'random_state': 42,
        'test_size': 0.25,
        'scaling': 'StandardScaler',
    }

    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(solver=params['solver'], max_iter=params['max_iter'], random_state=params['random_state'])),
    ])

    pipeline.fit(X_train, y_train)
    metrics, predictions = evaluate_classifier(pipeline, X_test, y_test)

    mlflow.log_params(params)
    mlflow.log_metrics(metrics)
    mlflow.set_tags({'team': 'sre-mlops-learning', 'owner_role': 'senior-sre', 'environment': 'local', 'chapter': 'mlflow-foundations', 'candidate_type': 'baseline', 'dataset_name': 'iris'})
    log_confusion_matrix_artifact(y_test, predictions, iris.target_names)
    mlflow.sklearn.log_model(sk_model=pipeline, artifact_path='model', input_example=X_test.head(3))

    print('Run ID:', run.info.run_id)
    print('Metrics:', metrics)


In [ ]:
with mlflow.start_run(run_name='random-forest-candidate') as run:
    params = {
        'model_type': 'RandomForestClassifier',
        'n_estimators': 100,
        'max_depth': 3,
        'random_state': 42,
        'test_size': 0.25,
        'scaling': 'none',
    }

    model = RandomForestClassifier(n_estimators=params['n_estimators'], max_depth=params['max_depth'], random_state=params['random_state'])
    model.fit(X_train, y_train)
    metrics, predictions = evaluate_classifier(model, X_test, y_test)

    mlflow.log_params(params)
    mlflow.log_metrics(metrics)
    mlflow.set_tags({'team': 'sre-mlops-learning', 'owner_role': 'senior-sre', 'environment': 'local', 'chapter': 'mlflow-foundations', 'candidate_type': 'candidate', 'dataset_name': 'iris'})
    log_confusion_matrix_artifact(y_test, predictions, iris.target_names)
    mlflow.sklearn.log_model(sk_model=model, artifact_path='model', input_example=X_test.head(3))

    print('Run ID:', run.info.run_id)
    print('Metrics:', metrics)


# 9. Compare runs and apply a quality gate

The UI is useful for humans, but automation needs programmatic access. A CI/CD pipeline can use this pattern to find the best run, check quality thresholds, block unsafe promotion, or register a candidate model.


In [ ]:
experiment = mlflow.get_experiment_by_name(experiment_name)
runs_df = mlflow.search_runs(experiment_ids=[experiment.experiment_id], order_by=['metrics.f1_macro DESC'])

columns_to_show = ['run_id', 'tags.mlflow.runName', 'params.model_type', 'metrics.accuracy', 'metrics.precision_macro', 'metrics.recall_macro', 'metrics.f1_macro', 'status', 'start_time']
display(runs_df[columns_to_show])

MIN_F1_MACRO = 0.90
model_runs_df = runs_df[runs_df['params.model_type'].notna()].copy()
model_runs_df = model_runs_df[model_runs_df['params.model_type'] != 'none']
best_run = model_runs_df.sort_values('metrics.f1_macro', ascending=False).iloc[0]

best_run_id = best_run['run_id']
best_run_name = best_run['tags.mlflow.runName']
best_model_type = best_run['params.model_type']
best_f1 = float(best_run['metrics.f1_macro'])

print('Best run:', best_run_name)
print('Best model type:', best_model_type)
print('Best run ID:', best_run_id)
print('Best F1 macro:', best_f1)
print('QUALITY GATE:', 'PASS' if best_f1 >= MIN_F1_MACRO else 'FAIL')


# 10. Load the selected model for inference

A model artifact is useful only if it can be loaded and used again. MLflow stores model artifacts with enough metadata to reload them through a model URI. The general format is `runs:/<run_id>/model`.


In [ ]:
model_uri = f'runs:/{best_run_id}/model'
loaded_model = mlflow.pyfunc.load_model(model_uri)

sample_input = X_test.head(5)
sample_predictions = loaded_model.predict(sample_input)

prediction_df = sample_input.copy()
prediction_df['prediction_id'] = sample_predictions
prediction_df['prediction_label'] = [iris.target_names[int(value)] for value in sample_predictions]
prediction_df


# 11. Production architecture

A production-grade MLflow setup separates compute, metadata, and artifacts:

```text
notebook / CI job / training pipeline
        |
        v
MLflow Tracking Server
        |
        +--> Backend store: PostgreSQL or MySQL
        |
        +--> Artifact store: S3 / GCS / Azure Blob / MinIO
        |
        +--> Model registry metadata
```

The backend store contains experiments, runs, parameters, metrics, tags, and registry metadata. The artifact store contains files such as model artifacts, plots, reports, and validation outputs. The tracking server exposes the API and UI and should be treated as an internal platform service.


# 12. SRE checklist for operating MLflow

## Availability
- run behind internal ingress or private load balancer;
- use readiness and liveness probes;
- alert on 5xx rate, latency, and availability;
- define an SLO for the tracking API.

## Persistence
- use durable database storage for metadata;
- use object storage for artifacts;
- avoid local pod disk for important artifacts;
- test backup and restore.

## Security
- protect the UI and API with authentication;
- use TLS;
- keep credentials in a secret manager;
- restrict object storage access;
- do not log secrets or sensitive raw data.

## Observability
- collect application logs;
- monitor request rate, error rate, and latency;
- monitor database connectivity;
- monitor artifact upload/download failures;
- monitor pod restarts and resource usage.


# 13. Runbook: MLflow tracking service degraded

## Symptoms
- UI unavailable;
- training jobs fail during `mlflow.log_*`;
- artifacts cannot be uploaded or downloaded;
- model registry operations fail;
- deployment pipeline cannot resolve model version.

## First checks
```bash
kubectl get pods -n mlops
kubectl get svc -n mlops
kubectl get ingress -n mlops
kubectl logs deploy/mlflow-tracking-server -n mlops --tail=100
```

## Backend checks
Check database reachability, credentials, connection limits, disk, migrations, and slow queries.

## Artifact checks
Check object storage reachability, IAM permissions, bucket policy, DNS, network policy, upload size limits, and timeout errors.

## Mitigation
Roll back recent config changes, restore previous secrets if rotation failed, restart unhealthy pods only after checking logs, pause model promotion if metadata integrity is uncertain, and verify recovery with a test run.


# 14. Interview-ready explanation

MLflow is an open-source platform for managing the machine learning lifecycle. It helps teams track experiments, log parameters and metrics, store artifacts, package models, and manage model versions. It improves reproducibility and traceability from experimentation to deployment.

A stronger Senior SRE version: MLflow is the metadata and artifact control plane around ML development. I would use it to connect training runs with parameters, metrics, artifacts, model versions, Git commits, CI build IDs, and deployment events. For production usage, I would run a remote tracking server with a durable database backend and object storage, protect it with authentication and TLS, monitor its availability and artifact operations, and define backup, restore, and model rollback procedures.


# 15. Practice assignments

1. Add a third model such as `KNeighborsClassifier` or `SVC` and log it to the same experiment.
2. Add tags for `git_commit`, `git_branch`, `ci_pipeline_id`, `author`, and `environment`.
3. Create and log a `model_card.md` artifact with intended use, dataset, metrics, limitations, risks, owner, and rollback plan.
4. Write a one-page design for production MLflow on Kubernetes with tracking server, PostgreSQL, object storage, ingress, TLS, authentication, secrets, monitoring, backup, and restore.
5. Define an SLO for the MLflow tracking API, including SLI, SLO, alert threshold, impact, and first response action.


# 16. Final summary

MLflow is foundational for MLOps because it gives structure, traceability, and reproducibility to model development. For local learning, a file-based tracking setup is fine. For team usage, MLflow should be operated as platform infrastructure with a tracking server, durable backend store, artifact store, authentication, monitoring, backups, and clear ownership.

The key SRE mindset is simple: a model is a production artifact. If it affects users or business decisions, it needs the same operational discipline as any production software artifact.
